# 第 3 章 §3.3 · Parallelization

三条**互不依赖**的链（摘要 / 提问 / 关键词）同时起飞，再汇合做一次综合。
形状是 **扇出 → 汇合**：前三章的原子操作（串行、分支、并发）到此凑齐。

kernel 选 `Python (agentic-book)`。只有【适配】处与书不同，prompt 和链结构未动。
本机实测加速比 **2.8~2.9x**（理论上限 3x）。七个关键点、执行时序、坑清单、LangGraph 并行拓扑都在 `../../my_notes/ch03_parallelization.md`。

In [1]:
import asyncio
import threading

# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from common import make_llm

# 【适配】原版：ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
# 只用一个 llm 对象被三条链共用 —— 没问题，每次 .invoke() 都是独立的 HTTP 请求。
llm = make_llm(temperature=0.7)


# 能并行的唯一前提：三条链的输入都是同一个 {topic}，谁也不吃谁的输出。
# 如果改成「把 summary 喂给 questions」，就必须串行（回到第 1 章）。
def make_chain(system_message: str):
    """同一个模板，只是 system 指令不同 —— 三条链的唯一差别就在这里。"""
    return (
        ChatPromptTemplate.from_messages([
            ("system", system_message),
            ("user", "{topic}"),
        ])
        | llm
        | StrOutputParser()
    )


summarize_chain = make_chain("Summarize the following topic concisely:")
questions_chain = make_chain("Generate three interesting questions about the following topic:")
terms_chain     = make_chain("Identify 5-10 key terms from the following topic, separated by commas:")


# ★ 本章新增 RunnableParallel：字典的键名 = 输出字段名，三条链同时收到同一份输入。
#
# ★ 为什么必须有 "topic": RunnablePassthrough() ？
#   RunnableParallel 的输出【只含它自己算出来的键】，原始输入默认被丢掉。
#   而下一步的 synthesis_prompt 需要 {topic}，少了这一行就 KeyError: 'topic'。
#   第 2 章也用过它（{"decision": ..., "request": RunnablePassthrough()}），同一个工具。
map_chain = RunnableParallel(
    summary=summarize_chain,
    questions=questions_chain,
    key_terms=terms_chain,
    topic=RunnablePassthrough(),
)

# 汇合点必然串行：要等三路都回来。
# 整条链耗时 ≈ max(三条链) + synthesis，而不是三者相加。
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
    Summary: {summary}
    Related Questions: {questions}
    Key Terms: {key_terms}
    Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}"),
])

full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


async def run_parallel_example(topic: str) -> None:
    """书里的原函数。注意它用的是 ainvoke（异步）。"""
    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    response = await full_parallel_chain.ainvoke({"topic": topic})
    print("\n--- Final Response ---")
    print(response)


def run_in_own_loop(async_fn):
    """【适配】在独立线程 + 独立事件循环里跑协程。

    书里直接写 asyncio.run(...)，在 .py 里没问题；但 notebook 本身已经在事件循环里，
    asyncio.run() 会报 "cannot be called from a running event loop"。
    ⚠️ 不逐个 asyncio.run：httpx 的异步客户端会绑定在第一个事件循环上，
       换循环就报 "Event loop is closed"。所以传进来的是「一次跑完所有异步调用」的函数。
    """
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(async_fn())
        except BaseException as e:            # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


# 书里是 `if __name__ == "__main__": asyncio.run(...)`。
# notebook 里 __name__ 也是 "__main__"，但事件循环已在运行，所以用上面的线程包装。
if __name__ == "__main__":
    run_in_own_loop(lambda: run_parallel_example("The history of space exploration"))


--- Running Parallel LangChain Example for Topic: 'The history of space exploration' ---



--- Final Response ---
Space exploration is best understood not as a simple chain of “firsts” but as the product of three overlapping forces: geopolitical competition, publicly funded science and engineering, and—more recently—commercial economics. The early rocket theorists and WWII-era missile technology made spaceflight possible; the **Space Race** made it urgent; and the post-Cold War era made it increasingly international and commercial.

## 1. The historical arc

The story begins with early rocket theory and wartime technology, then accelerates in the Cold War. The Soviet Union’s **Sputnik 1** in 1957 was the first artificial satellite and a psychological shock to the United States. It helped trigger the creation of **NASA** in 1958 and a broad American investment in science, engineering, and missile/space technology. The Soviet Union followed with **Yuri Gagarin** in 1961, the first human in space. The U.S. response culminated in the **Apollo 11** mission of 1969, when humans f